In [ ]:
# 1.1: Start with a fresh clone of YOUR repository.
!rm -rf /kaggle/working/MeloTTS-RO
!git clone https://github.com/CrisChir/MeloTTS-RO.git

In [ ]:


# 1.2: Install all dependencies from your repository.
!pip install -q -e /kaggle/working/MeloTTS-RO/

# 1.3: Manually download the Japanese dictionary data for MeCab to prevent errors.
!python -m unidic download

print("✅ Environment setup complete.")

In [ ]:
import os

from tqdm import tqdm

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
# Set to True to process only a small number of files for quick testing.
# Set to False to process the entire dataset for the final training run.
# TEST_MODE = False
TEST_MODE = True
NUM_TEST_FILES = 10000  # The number of files to process in test mode.
# ==============================================================================

# --- File Paths ---
INPUT_METADATA_PATH = '/kaggle/input/melotts-dataset-romanian/consolidated_melotts_data/metadata.list'
OUTPUT_METADATA_PATH = '/kaggle/working/metadata_corrected.list'
PATH_PREFIX = '/kaggle/input/melotts-dataset-romanian/consolidated_melotts_data/'

# --- Script Logic ---
modified_lines = []

print(f"Reading from: {INPUT_METADATA_PATH}")
with open(INPUT_METADATA_PATH, 'r', encoding='utf-8') as f:
    all_lines = f.readlines()

lines_to_process = all_lines

if TEST_MODE:
    print(f"\n--- RUNNING IN TEST MODE ---")
    print(f"Processing only the first {NUM_TEST_FILES} files.")
    lines_to_process = all_lines[:NUM_TEST_FILES]
else:
    print(f"\n--- RUNNING IN FULL MODE ---")
    print(f"Processing all {len(all_lines)} files.")

for line in tqdm(lines_to_process, desc='Correcting metadata'):
    line = line.strip()
    if not line: continue
    
    parts = line.split('|', 3)
    if len(parts) < 4: continue
    
    # Prepend the absolute path
    parts[0] = os.path.join(PATH_PREFIX, parts[0])
    
    # Ensure language code is uppercase
    if parts[2] == 'ro':
        parts[2] = 'RO'
        
    modified_lines.append('|'.join(parts))

with open(OUTPUT_METADATA_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(modified_lines))
    
print(f'\n✅ SUCCESS: Corrected metadata saved to {OUTPUT_METADATA_PATH}')
print(f"Total lines processed and saved: {len(modified_lines)}")

In [ ]:
%env TOKENIZERS_PARALLELISM=false

!python /kaggle/working/MeloTTS-RO/melo/preprocess_text.py \
    --metadata /kaggle/working/metadata_corrected.list \
    --config_path /kaggle/working/MeloTTS-RO/melo/configs/config.json

In [ ]:
import torch
from tqdm import tqdm
import os
from collections import defaultdict
import itertools

metadata_path = "/kaggle/working/metadata_corrected.list.cleaned"
bert_root = "/kaggle/working/bert_features"
delete_mismatched = True
skip_missing = False  # Set to True if you want to ignore missing .bert.pt files

def validate_bert_files():
    broken = []
    error_types = defaultdict(int)

    with open(metadata_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in tqdm(lines, desc="Validating BERT files"):
        parts = line.strip().split("|")
        if len(parts) < 7:
            error_types["malformed_line"] += 1
            continue

        wav_path = parts[0]
        phones = parts[4].split(" ")
        tone = parts[5].split(" ")
        word2ph = list(map(int, parts[6].split(" ")))

        # Apply blank expansion logic
        phones = [0] + list(itertools.chain.from_iterable(zip(phones, [0]*len(phones))))
        tone = [0] + list(itertools.chain.from_iterable(zip(tone, [0]*len(tone))))
        word2ph = [w * 2 for w in word2ph]
        word2ph[0] += 1

        expected_len = len(phones)
        rel_path = os.path.relpath(wav_path, "/kaggle/input/")
        bert_path = os.path.join(bert_root, rel_path).replace(".wav", ".bert.pt")

        if not os.path.exists(bert_path):
            error_types["missing_file"] += 1
            if not skip_missing:
                broken.append((bert_path, "missing", expected_len))
            continue

        try:
            bert = torch.load(bert_path)
            actual_len = bert.shape[1] if bert.ndim in [2, 3] else -1

            if actual_len != expected_len:
                error_types["length_mismatch"] += 1
                broken.append((bert_path, actual_len, expected_len))
                if delete_mismatched:
                    os.remove(bert_path)
        except Exception as e:
            error_types["load_error"] += 1
            broken.append((bert_path, "load_error", str(e)))
            if delete_mismatched:
                os.remove(bert_path)

    print(f"\n✅ Scan complete. Found {len(broken)} broken BERT files.")
    for path, actual, expected in broken:
        print(f"❌ {path} → shape[1]={actual}, expected={expected}")

    print("\n📊 Error Summary:")
    for err_type, count in error_types.items():
        print(f"  - {err_type}: {count}")

validate_bert_files()

In [ ]:
!cd /kaggle/working/MeloTTS-RO/MeloTTS-RO

In [ ]:
ls

In [ ]:
# import argparse
# import json

# def get_hparams():
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--config", type=str, default="/kaggle/working/MeloTTS-RO/confis/config.json")
#     args, _ = parser.parse_known_args()
#     with open(args.config, "r") as f:
#         data = json.load(f)
#     return HParams(**data)

In [ ]:
# import time
# start = time.time()
# # Run 100 steps
# # # !python /kaggle/working/MeloTTS-RO/melo/train.py --config /kaggle/working/MeloTTS-RO/configs/base.json -m ro_model
# # !python /kaggle/working/MeloTTS-RO/melo/train.py --config /kaggle/working/MeloTTS-RO/configs/config.json -m ro_model
# !python /kaggle/working/MeloTTS-RO/melo/train.py --config /kaggle/working/config.json -m ro_model


# # !python /kaggle/working/MeloTTS-RO/melo/train.py --config kaggle/working/MeloTTS-RO/melo/configs/config.json  -m ro_model
# elapsed = time.time() - start
# steps_per_hour = 3600 / (elapsed / 100)


In [ ]:
!cd kaggle/working/MeloTTS-RO/configs/

In [ ]:
ls

In [ ]:
# import time
# start = time.time()
# !python /kaggle/working/MeloTTS-RO/melo/train.py --config configs/base.yaml -m ro_model
# elapsed = time.time() - start
# steps_per_hour = 3600 / (elapsed / 100)

In [ ]:
%%bash
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True



In [ ]:
import time
start = time.time()
!cd /kaggle/working/MeloTTS-RO && \
torchrun --nproc_per_node=2 melo/train.py -c /kaggle/working/config.json -m ro_model
elapsed = time.time() - start
steps_per_hour = 3600 / (elapsed / 770)
print("steps_per_hour",steps_per_hour)

In [ ]:
# !cd /kaggle/working/MeloTTS-RO && \
# torchrun --nproc_per_node=2 -m melo.train -c /kaggle/working/config.json -m ro_model

In [ ]:
# python infer.py \
#   --text "Bună ziua, cum vă pot ajuta?" \
#   -m /kaggle/working/checkpoints/G_14000.pth \
#   -l RO \
#   -o /kaggle/working/infer_outputs


In [ ]:
!python /kaggle/working/MeloTTS-RO/melo/infer_ro.py \
  --text "Acesta este un test pentru vocea sintetică." \
  -m logs/ro_model \
  -l RO \
  -o infer_outputs


In [ ]:
!python /kaggle/working/MeloTTS-RO/melo/infer_ro.py \
  --text "Acesta este un test pentru vocea sintetică." \
  -m /kaggle/working/MeloTTS-RO/logs/ro_model \
  -l RO \
  -o infer_outputs

In [ ]:
!ls /kaggle/working/MeloTTS-RO/logs/ro_model


In [ ]:
# import glob
# glob.glob("/kaggle/**/ro_model", recursive=True)

In [ ]:
!ls /kaggle/working/MeloTTS-RO/logs/ro_model/G_*.pth


In [ ]:
ls -lh /kaggle/working/MeloTTS-RO/logs/ro_model/config.json
